# ディズニーランド・リゾート 待ち時間分析ノート

`python warehouse.py` でデータベースを作ってから使います。上から順に実行してください(Shift+Enter)。
どのセルも **SQLを書き換えれば切り口を変えられる** ようにしてあります。最後のセルは自由に書く用です。

使えるテーブル: `fact_waits`(記録)/ `dim_ride`(アトラクション)/ `dim_calendar`(日付・混雑指数)/ `dim_show`(ショー)
ビュー: `v_slot_profile` / `v_daily_ride` / `v_downtime_events` / `v_queue_people`

Powered by [Queue-Times.com](https://queue-times.com/)

In [ ]:
import duckdb, pandas as pd, numpy as np, matplotlib.pyplot as plt
from matplotlib import font_manager

# 日本語フォント(Mac / Windows / Linux の順に探す)
for f in ["Hiragino Sans", "Hiragino Maru Gothic Pro", "Yu Gothic", "Meiryo", "Noto Sans CJK JP", "Noto Sans JP", "IPAexGothic"]:
    if any(f == x.name for x in font_manager.fontManager.ttflist):
        plt.rcParams["font.family"] = f; break
plt.rcParams.update({"figure.figsize": (11, 4.5), "axes.spines.top": False, "axes.spines.right": False,
                     "axes.grid": True, "grid.color": "#e6e5e0", "axes.axisbelow": True, "figure.dpi": 110})
COLORS = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#6250d6", "#e34948"]  # 色の順番は固定

con = duckdb.connect("warehouse/disney.duckdb", read_only=True)
q = lambda sql: con.sql(sql).df()      # SQL → pandas の表
PARK = "DL"                             # "DL" か "DCA"
q("SELECT park, count(DISTINCT date_local) AS 日数, min(date_local) AS 最初, max(date_local) AS 最後, count(*) AS 記録数 FROM fact_waits GROUP BY park")

## 1. 日ごとの混雑指数
100 = データ期間の平均的な日。人気アトラクション(平均40分以上)の日平均待ち ÷ 全期間平均。

In [ ]:
df = q("SELECT park, date_local, crowd_index, is_weekend, holiday FROM dim_calendar ORDER BY date_local")
fig, ax = plt.subplots()
for i, (p, g) in enumerate(df.groupby("park")):
    ax.plot(g.date_local, g.crowd_index, marker="o", lw=2, color=COLORS[i], label=p)
ax.axhline(100, color="#9a9890", ls="--", lw=1); ax.set_ylabel("混雑指数"); ax.legend(frameon=False); fig.autofmt_xdate(); plt.show()
df.pivot(index="date_local", columns="park", values="crowd_index").round(0)

## 2. アトラクション × 時間帯の待ち時間(ヒートマップ)
15分ごとの中央値。色が濃いほど長い。

In [ ]:
df = q(f"""SELECT name, slot15, median_wait FROM v_slot_profile
          WHERE park = '{PARK}' AND name IN (SELECT name FROM v_slot_profile WHERE park = '{PARK}' GROUP BY name ORDER BY avg(median_wait) DESC LIMIT 20)""")
pv = df.pivot_table(index="name", columns="slot15", values="median_wait")
pv = pv.loc[pv.mean(axis=1).sort_values(ascending=False).index]
fig, ax = plt.subplots(figsize=(13, 7))
im = ax.imshow(pv.values, aspect="auto", cmap="Blues")
ax.set_yticks(range(len(pv.index)), pv.index, fontsize=9)
ticks = [i for i, s in enumerate(pv.columns) if s % 120 == 0]
ax.set_xticks(ticks, [f"{pv.columns[i] // 60}時" for i in ticks]); ax.grid(False)
fig.colorbar(im, ax=ax, label="待ち時間(分)"); plt.show()

## 3. 平日と週末の比較
人気アトラクションの平均待ちを、時間帯ごとに。

In [ ]:
df = q(f"""SELECT c.is_weekend, f.slot15, avg(f.wait_time) AS w
          FROM fact_waits f JOIN dim_calendar c USING (park, date_local)
          WHERE f.park = '{PARK}' AND f.ride_id IN (SELECT ride_id FROM fact_waits WHERE park = '{PARK}' GROUP BY ride_id HAVING avg(wait_time) >= 40)
          GROUP BY ALL ORDER BY slot15""")
fig, ax = plt.subplots()
for i, (k, g) in enumerate(df.groupby("is_weekend")):
    ax.plot(g.slot15 / 60, g.w, lw=2, color=COLORS[i], label="週末" if k else "平日")
ax.set_xlabel("時刻"); ax.set_ylabel("平均待ち(分)"); ax.legend(frameon=False); plt.show()

## 4. 混雑指数への反応
アトラクションの1日平均待ちと、その日の混雑指数の関係。傾きが大きいほど、混む日に一気に伸びる。

In [ ]:
df = q(f"""SELECT d.name, d.date_local, d.avg_wait, c.crowd_index
          FROM v_daily_ride d JOIN dim_calendar c USING (park, date_local) WHERE d.park = '{PARK}' AND d.avg_wait IS NOT NULL""")
res = []
for name, g in df.groupby("name"):
    if len(g) >= 3 and g.crowd_index.std() > 0:
        b, a = np.polyfit(g.crowd_index, g.avg_wait, 1); r = np.corrcoef(g.crowd_index, g.avg_wait)[0, 1]
        res.append({"アトラクション": name, "指数100の平均待ち": round(a + b * 100), "指数+10で何分増えるか": round(b * 10, 1), "R²": round(r ** 2, 2), "日数": len(g)})
sens = pd.DataFrame(res).sort_values("指数+10で何分増えるか", ascending=False).reset_index(drop=True)
top = sens.head(4)["アトラクション"].tolist()
fig, ax = plt.subplots()
for i, n in enumerate(top):
    g = df[df.name == n]; ax.scatter(g.crowd_index, g.avg_wait, color=COLORS[i], s=40, label=n)
    b, a = np.polyfit(g.crowd_index, g.avg_wait, 1); xs = np.array([g.crowd_index.min(), g.crowd_index.max()]); ax.plot(xs, a + b * xs, color=COLORS[i], lw=2)
ax.set_xlabel("混雑指数"); ax.set_ylabel("1日の平均待ち(分)"); ax.legend(frameon=False, fontsize=9); plt.show()
sens

## 5. ショー中に空くアトラクション
ショーの時間帯の平均待ちが、直前30分と比べて何%変わったか(`shows.csv` の時刻を使用)。マイナスほど空く。

In [ ]:
df = q(f"""WITH x AS (
  SELECT s.show_name, f.name,
         avg(f.wait_time) FILTER (WHERE f.slot15 >= s.start_min AND f.slot15 < s.end_min) AS during,
         avg(f.wait_time) FILTER (WHERE f.slot15 >= s.start_min - 30 AND f.slot15 < s.start_min) AS before
  FROM fact_waits f JOIN dim_show s ON s.park = f.park AND s.date_local = f.date_local
  WHERE f.park = '{PARK}' GROUP BY ALL)
SELECT show_name AS ショー, name AS アトラクション, round(before) AS 直前, round(during) AS ショー中, round(100 * (during / NULLIF(before, 0) - 1)) AS 変化率
FROM x WHERE before >= 15 ORDER BY ショー, 変化率""")
df.groupby("ショー").head(5)

## 6. システム調整(シス調)
営業中に止まった記録。1日に1回以上止まる確率と、止まり始める時間帯。

In [ ]:
days = q(f"SELECT count(*) AS n FROM dim_calendar WHERE park = '{PARK}'").n[0]
ev = q(f"""SELECT name, count(*) AS 回数, count(DISTINCT date_local) AS 止まった日数,
              round(avg(date_diff('minute', down_from, up_again))) AS 平均停止分
          FROM v_downtime_events WHERE park = '{PARK}' AND up_again IS NOT NULL GROUP BY name""")
ev["1日に1回以上止まる確率"] = (ev.止まった日数 / days).round(2); ev["1日あたり回数"] = (ev.回数 / days).round(2)
ev = ev.sort_values("1日に1回以上止まる確率", ascending=False)
hh = q(f"SELECT name, start_hour, count(*) AS n FROM v_downtime_events WHERE park = '{PARK}' GROUP BY ALL")
pv = hh.pivot_table(index="name", columns="start_hour", values="n", fill_value=0).loc[ev.name.head(15)] / days
fig, ax = plt.subplots(figsize=(12, 5.5)); im = ax.imshow(pv.values, aspect="auto", cmap="Blues")
ax.set_yticks(range(len(pv.index)), pv.index, fontsize=9); ax.set_xticks(range(len(pv.columns)), [f"{h}時" for h in pv.columns]); ax.grid(False)
fig.colorbar(im, ax=ax, label="1日あたりの停止開始回数"); plt.show()
ev.head(15)

## 7. ゲスト占有率(列に並んでいる人の割合)
リトルの法則で推定した列の人数のうち、各アトラクションが占める割合。上位6件+その他。

In [ ]:
df = q(f"SELECT name, slot15, people_in_line, share FROM v_queue_people WHERE park = '{PARK}'")
top = df.groupby("name").people_in_line.sum().sort_values(ascending=False).head(6).index.tolist()
df["grp"] = df.name.where(df.name.isin(top), "その他")
pv = df.pivot_table(index="slot15", columns="grp", values="people_in_line", aggfunc="sum").fillna(0)[top + ["その他"]]
fig, ax = plt.subplots()
ax.stackplot(pv.index / 60, pv.T.values, labels=pv.columns, colors=COLORS[:6] + ["#9a9890"], edgecolor="white", linewidth=0.8)
ax.set_xlabel("時刻"); ax.set_ylabel("列の人数(推定)"); ax.legend(frameon=False, fontsize=8, loc="upper left", ncol=2); plt.show()
(df.pivot_table(index="name", columns=(df.slot15 // 120 * 2).astype(str) + "時", values="share", aggfunc="mean") * 100).round(1).loc[top]

## 8. 一緒に混むアトラクション(相関)
日ごとの平均待ちの相関。1に近いほど同じように増減する。

In [ ]:
d = q(f"SELECT date_local, name, avg_wait FROM v_daily_ride WHERE park = '{PARK}'")
top = d.groupby("name").avg_wait.mean().sort_values(ascending=False).head(15).index
cm = d[d.name.isin(top)].pivot(index="date_local", columns="name", values="avg_wait").corr()
fig, ax = plt.subplots(figsize=(9, 8)); im = ax.imshow(cm.values, cmap="RdBu_r", vmin=-1, vmax=1)
ax.set_xticks(range(len(cm)), cm.columns, rotation=90, fontsize=8); ax.set_yticks(range(len(cm)), cm.index, fontsize=8); ax.grid(False)
fig.colorbar(im, ax=ax, label="相関係数"); plt.show()

## 9. 朝いちばんに乗る価値
開園30分の平均待ちと、昼(11〜17時)の平均待ちの差。

In [ ]:
q(f"""WITH o AS (SELECT park, date_local, CAST(split_part(open_est, ':', 1) AS INT) * 60 + CAST(split_part(open_est, ':', 2) AS INT) AS om FROM dim_calendar)
SELECT f.name AS アトラクション,
       round(avg(f.wait_time) FILTER (WHERE f.slot15 < o.om + 30)) AS 開園30分,
       round(avg(f.wait_time) FILTER (WHERE f.slot15 BETWEEN 660 AND 1019)) AS 昼,
       round(avg(f.wait_time) FILTER (WHERE f.slot15 BETWEEN 660 AND 1019) - avg(f.wait_time) FILTER (WHERE f.slot15 < o.om + 30)) AS 得する分
FROM fact_waits f JOIN o USING (park, date_local)
WHERE f.park = '{PARK}' GROUP BY ALL ORDER BY 得する分 DESC LIMIT 20""")

## 自由にSQL
例:12月の土曜だけ、17時以降のスペース・マウンテンの待ち時間

In [ ]:
q("""
SELECT date_local, ts_local, wait_time
FROM fact_waits
WHERE park = 'DL' AND ride_no = 51 AND dayname(date_local) = 'Saturday' AND hour_local >= 17
ORDER BY ts_local
""")